In [341]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import accuracy_score, log_loss
from IPython.display import display

    
PROJECT_DIR = Path.cwd()
DATA_FILE = PROJECT_DIR / 'Stock_Portfolio_Raw_Adjusted_yfinance.csv'
GLOBAL_FILE = PROJECT_DIR / 'Global_Indicators_yfinance.csv'

TRAIN_END = pd.Timestamp('2020-12-31')
TUNING_START = pd.Timestamp('2021-01-04')
TUNING_END = pd.Timestamp('2021-12-31')

H = 5
SEED = 42
TICKERS = ['AAPL', 'XOM', 'JNJ']

MAX_PILOT_STEPS = 12
TRAIN_WINDOW = 160 
PROBABILITY_THRESHOLD = 0.5
RETRAIN_TREES = 80
FINETUNE_TREES = 15 

TRANSACTION_BPS = 5.0 
TRANSACTION_RATE = TRANSACTION_BPS / 10_000

REFRESH_FROM_YFINANCE = False   
SAVE_DOWNLOADED_DATA = False 

In [342]:

import yfinance as yf
def download_daily(symbols, start='2020-01-01', end='2026-01-01'):
    frames = []
    for symbol in symbols:
        part = yf.download(
            symbol, start=start, end=end, interval='1d',
            auto_adjust=False, actions=True, multi_level_index=False,
            threads=False, progress=False,
        )
        part = part.rename_axis('Date').reset_index()
        part['Ticker'] = symbol
        frames.append(part)
    return pd.concat(frames, ignore_index=True).sort_values(['Ticker', 'Date'])

if REFRESH_FROM_YFINANCE:
    core_tickers = ['AAPL', 'AMZN', 'JPM', 'JNJ', 'XOM', 'CAT', 'NVDA', 'MSFT']
    downloaded_stocks = download_daily(core_tickers)
    downloaded_globals = download_daily(['^VIX', '^GSPC', '^TNX'])
    display(downloaded_stocks.groupby('Ticker')['Date'].agg(['min', 'max', 'count']))
    if SAVE_DOWNLOADED_DATA:
        downloaded_stocks.to_csv(
            DATA_FILE, index=False, mode='x',
        )
        downloaded_globals.to_csv(
            GLOBAL_FILE, index=False, mode='x',
        )


In [343]:
def load_market_data(path, tickers):
    df = pd.read_csv(path)
    numeric = ['Open', 'High', 'Low', 'Close', 'Volume']
    prices = ['Open', 'High', 'Low', 'Close']

    df['Date'] = pd.to_datetime(df['Date'], errors='raise', utc=True).dt.tz_localize(None).dt.normalize()
    missing_tickers = set(tickers) - set(df['Ticker'])
    
    if missing_tickers:
        raise ValueError(f'Нет запрошенных активов: {missing_tickers}')
    
    if 'Adj Close' in df.columns:
        numeric.append('Adj Close')
    
    df[numeric] = df[numeric].apply(pd.to_numeric, errors='raise')
    if (df[numeric] <= 0).any().any():
        raise ValueError('Неположительная цена или объём в выбранных акциях.')
    
    factor = df['Adj Close'] / df['Close']
    df[prices] = df[prices].mul(factor, axis=0)
    df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    panels = {col: df.pivot(index='Date', columns='Ticker', values=col).sort_index().reindex(columns=tickers) for col in numeric}
    return df, panels

stock_data, panels = load_market_data(DATA_FILE, TICKERS)
open_px, close_px = panels['Open'], panels['Close']
dates = close_px.index

display(stock_data.groupby('Ticker')['Date'].agg(['min', 'max', 'count']))

,min,max,count
Ticker,,,
AAPL,2020-01-02,2025-12-31,1508
AMZN,2020-01-02,2025-12-31,1508
CAT,2020-01-02,2025-12-31,1508
JNJ,2020-01-02,2025-12-31,1508
JPM,2020-01-02,2025-12-31,1508
MSFT,2020-01-02,2025-12-31,1508
NVDA,2020-01-02,2025-12-31,1508
XOM,2020-01-02,2025-12-31,1508


In [344]:
def build_features(price_panels):
    features = {}
    for ticker in price_panels['Close'].columns:
        close = price_panels['Close'][ticker]
        log_return = np.log(close).diff() 
        frame = pd.DataFrame(index=close.index)
        frame['mom_5'] = log_return.rolling(5).sum()
        frame['mom_20'] = log_return.rolling(20).sum()

        frame['vol_20'] = log_return.rolling(20).std()
        long_vol = log_return.rolling(60).std().replace(0, np.nan)

        frame['vol_ratio'] = log_return.rolling(5).std() / long_vol
        frame['ma_ratio'] = close / close.rolling(20).mean() - 1

        delta = close.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)

        period = 14

        avg_gain = gain.ewm(
            alpha=1 / period,
            min_periods=period,
            adjust=False
        ).mean()

        avg_loss = loss.ewm(
            alpha=1 / period,
            min_periods=period,
            adjust=False
        ).mean()

        frame['rsi_14'] = 100 - 100 / (1 + avg_gain / avg_loss.replace(0, np.nan)).replace(0, np.nan)
        frame.loc[(gain==0) & (loss == 0), 'rsi_14'] = 50 

        frame['day_range'] = (price_panels['High'][ticker] - price_panels['Low'][ticker]) / close
        frame['day_body'] = close / price_panels['Open'][ticker] - 1
        if np.isinf(frame.to_numpy()).any():
            raise ValueError(f'Бесконечный признак у {ticker}.')
        features[ticker] = frame

    return features
    
features = build_features(panels)

In [345]:
def build_targets(open_prices, horizon):
    calendar = pd.Series(open_prices.index, index=open_prices.index)
    entry_at = calendar.shift(-1)
    available_at = calendar.shift(-(horizon + 1))
    returns = open_prices.shift(-(horizon + 1)) / open_prices.shift(-1) - 1
    targets = {}
    for ticker in open_prices.columns:
        future_return = returns[ticker]
        targets[ticker] = pd.DataFrame({
            'forward_return': future_return,
            'label': (future_return > 0).astype(float).where(future_return.notna()),
            'entry_at': entry_at,
            'available_at': available_at,
        })
    return targets

targets = build_targets(open_px, H)

def training_data(ticker, as_of):
    # as_of — дата, к закрытию которой обучение разрешено знать ответы.
    X, target = features[ticker], targets[ticker]
    eligible = (
        (X.index <= as_of) & X.notna().all(axis=1)
        & target['label'].notna() & (target['available_at'] <= as_of)
    )
    X_train = X.loc[eligible].tail(TRAIN_WINDOW)
    y_train = target.loc[X_train.index, 'label'].astype(int)
    last_label_available = target.loc[X_train.index, 'available_at'].max()
    assert last_label_available <= as_of  # Защитная проверка отсутствия будущих ответов.
    return X_train, y_train, last_label_available

X_example, y_example, label_cutoff = training_data('AAPL', TRAIN_END)
print('Начальное обучение: строк', len(X_example))
print('Последняя дата признаков:', X_example.index.max().date())
print('Последний ответ стал доступен:', label_cutoff.date())
X_example.head()

Начальное обучение: строк 160
Последняя дата признаков: 2020-12-22
Последний ответ стал доступен: 2020-12-31


,mom_5,mom_20,vol_20,vol_ratio,ma_ratio,rsi_14,day_range,day_body
Date,,,,,,,,
2020-05-07,0.033272,0.132412,0.020536,0.288024,0.067805,64.123460,0.010535,0.001715
2020-05-08,0.073026,0.148745,0.020882,0.120319,0.085024,67.182577,0.019540,0.014690
2020-05-11,0.074589,0.144920,0.020783,0.120319,0.093965,69.102849,0.031142,0.022428
2020-05-12,0.048197,0.084157,0.018648,0.288310,0.076763,66.033122,0.028194,-0.020199
2020-05-13,0.025786,0.081179,0.018772,0.359579,0.059369,62.890831,0.041411,-0.014416


In [346]:
def base_params(n_trees, seed=SEED):
    return {
        'n_estimators': n_trees,
        'learning_rate': 0.05,  # Шаг каждого нового дерева.
        'num_leaves': 7,  # Небольшие деревья для небольшой учебной выборки.
        'min_child_samples': 20,
        'subsample': 0.9,
        'subsample_freq': 1, 
        'colsample_bytree': 0.9, 
        'random_state': seed,
        'n_jobs': 1,  
        'deterministic': True,
        'force_col_wise': True,
        'verbosity': -1,
    }

def action_retrain(X, y, seed=SEED):
    model = lgb.LGBMClassifier(**base_params(RETRAIN_TREES, seed))
    model.fit(X, y)
    return model

def action_fine_tune(current_model, X, y, seed=SEED):
    # Новый объект сохраняет исходную модель для честного сравнения кандидатов.
    model = lgb.LGBMClassifier(**base_params(FINETUNE_TREES, seed))
    model.fit(X, y, init_model=current_model.booster_)
    return model

def action_no_op(current_model):
    return current_model

initial_model = action_retrain(X_example, y_example)
print('Деревьев в начальной модели:', initial_model.booster_.num_trees())

Деревьев в начальной модели: 80


In [347]:
def make_schedule(calendar, start, end, horizon, max_steps):
    decision_idx = int(calendar.searchsorted(start))
    rows = []

    while len(rows) < max_steps:

        entry_idx = decision_idx + 1
        exit_idx = decision_idx + horizon + 1

        if calendar[exit_idx] > end:
            break
        rows.append({'decision_date': calendar[decision_idx], 'entry_date': calendar[entry_idx], 'exit_date': calendar[exit_idx]})
        if len(rows) >= max_steps:
            break
        decision_idx = exit_idx

    return pd.DataFrame(rows)

schedule = make_schedule(dates, TUNING_START, TUNING_END, H, MAX_PILOT_STEPS)
print(schedule)
print('Интервалов:', len(schedule), '| Последний выход:', schedule['exit_date'].max().date())
distinctness_rows = []

first_decision = schedule.iloc[0]['decision_date']
for ticker in TICKERS:
    
    X_old, y_old, _ = training_data(ticker, TRAIN_END)
    X_new, y_new, _ = training_data(ticker, first_decision)

    deployed = action_retrain(X_old, y_old)
    old_tree = deployed.booster_.num_trees()
    candidates = {
        'no-op': action_no_op(deployed),
        'fine-tune': action_fine_tune(deployed, X_new, y_new),
        'retrain': action_retrain(X_new, y_new),
    }

    x_now = features[ticker].loc[[first_decision]]
    for action, candidate in candidates.items():
        probability = float(candidate.predict_proba(x_now)[0, 1])
        distinctness_rows.append({
                'ticker': ticker, 'action': action, 'probability': probability,
                'signal': int(probability > PROBABILITY_THRESHOLD),
                'n_trees': candidate.booster_.num_trees(),
            })
    assert deployed.booster_.num_trees() == old_tree  # Исходная модель не изменена.
distinctness = pd.DataFrame(distinctness_rows)
display(distinctness)


   decision_date entry_date  exit_date
0     2021-01-04 2021-01-05 2021-01-12
1     2021-01-12 2021-01-13 2021-01-21
2     2021-01-21 2021-01-22 2021-01-29
3     2021-01-29 2021-02-01 2021-02-08
4     2021-02-08 2021-02-09 2021-02-17
5     2021-02-17 2021-02-18 2021-02-25
6     2021-02-25 2021-02-26 2021-03-05
7     2021-03-05 2021-03-08 2021-03-15
8     2021-03-15 2021-03-16 2021-03-23
9     2021-03-23 2021-03-24 2021-03-31
10    2021-03-31 2021-04-01 2021-04-09
11    2021-04-09 2021-04-12 2021-04-19
Интервалов: 12 | Последний выход: 2021-04-19


,ticker,action,probability,signal,n_trees
0,AAPL,no-op,0.939076,1,80
1,AAPL,fine-tune,0.945613,1,95
2,AAPL,retrain,0.903707,1,80
3,XOM,no-op,0.194026,0,80
4,XOM,fine-tune,0.155149,0,95
5,XOM,retrain,0.168689,0,80
6,JNJ,no-op,0.603328,1,80
7,JNJ,fine-tune,0.674017,1,95
8,JNJ,retrain,0.677379,1,80


In [348]:
def choose_demo_action(policy_name, step):
    if policy_name == 'never_retrain':
        return 'no-op'
    if policy_name == 'weekly_retrain':
        return 'retrain'
    if policy_name == 'demo_cycle':
        return ('no-op', 'fine-tune', 'retrain')[step % 3]

def run_pilot(policy_name, seed=SEED):
    states, initial_times, rows = {}, {}, []
    for ticker in TICKERS:
        X, y, last_available = training_data(ticker, TRAIN_END)
        started = time.perf_counter()
        model = action_retrain(X, y, seed)
        initial_times[ticker] = time.perf_counter() - started

        states[ticker] = {
            'model': model, 'version': 0, 'last_update': TRAIN_END,
            'last_retrain': TRAIN_END, 'n_train': len(X),
            'last_train_origin': X.index.max(), 'last_label_available': last_available,
        }
    for step, interval in schedule.iterrows():
        decision_date = interval['decision_date']
        actions = {ticker: choose_demo_action(policy_name, step) for ticker in TICKERS}
        for ticker in TICKERS:
            state, action = states[ticker], actions[ticker]
            x_now = features[ticker].loc[[decision_date]]
            X, y, last_available = training_data(ticker, decision_date)
            started = time.perf_counter()
            if action == 'retrain':
                state['model'] = action_retrain(X, y, seed)
                state['last_retrain'] = decision_date
            elif action == 'fine-tune':
                state['model'] = action_fine_tune(state['model'], X, y, seed)
            else:
                state['model'] = action_no_op(state['model'])
            seconds = time.perf_counter() - started
            if action != 'no-op':
                state.update({
                    'version': state['version'] + 1, 'last_update': decision_date,
                    'n_train': len(X), 'last_train_origin': X.index.max(),
                    'last_label_available': last_available,
                })
            probability = float(state['model'].predict_proba(x_now)[0, 1])
            rows.append({
                'policy': policy_name, 'ticker': ticker, 'step': step,
                'decision_date': decision_date, 'entry_date': interval['entry_date'],
                'exit_date': interval['exit_date'], 'action': action,
                'probability': probability, 'signal': int(probability > PROBABILITY_THRESHOLD),
                'version': state['version'], 'n_trees': state['model'].booster_.num_trees(),
                'n_train': state['n_train'], 'last_train_origin': state['last_train_origin'],
                'last_label_available': state['last_label_available'],
                'days_since_retrain': (decision_date - state['last_retrain']).days,
                'seconds': seconds,
            })
    return pd.DataFrame(rows), initial_times
        

POLICIES = ['never_retrain', 'weekly_retrain', 'demo_cycle']

POLICIES = ['never_retrain', 'weekly_retrain', 'demo_cycle']
decision_logs, initial_training_times = {}, {}
for policy_name in POLICIES:
    decision_logs[policy_name], initial_training_times[policy_name] = run_pilot(policy_name)
    print(policy_name, ':', len(decision_logs[policy_name]), 'решений')
# display(decision_logs['demo_cycle'].head(9))
# display(decision_logs['demo_cycle'])

never_retrain : 36 решений
weekly_retrain : 36 решений
demo_cycle : 36 решений


In [349]:
decision_logs

{'never_retrain':            policy ticker  step decision_date entry_date  exit_date action  \
 0   never_retrain   AAPL     0    2021-01-04 2021-01-05 2021-01-12  no-op   
 1   never_retrain    XOM     0    2021-01-04 2021-01-05 2021-01-12  no-op   
 2   never_retrain    JNJ     0    2021-01-04 2021-01-05 2021-01-12  no-op   
 3   never_retrain   AAPL     1    2021-01-12 2021-01-13 2021-01-21  no-op   
 4   never_retrain    XOM     1    2021-01-12 2021-01-13 2021-01-21  no-op   
 5   never_retrain    JNJ     1    2021-01-12 2021-01-13 2021-01-21  no-op   
 6   never_retrain   AAPL     2    2021-01-21 2021-01-22 2021-01-29  no-op   
 7   never_retrain    XOM     2    2021-01-21 2021-01-22 2021-01-29  no-op   
 8   never_retrain    JNJ     2    2021-01-21 2021-01-22 2021-01-29  no-op   
 9   never_retrain   AAPL     3    2021-01-29 2021-02-01 2021-02-08  no-op   
 10  never_retrain    XOM     3    2021-01-29 2021-02-01 2021-02-08  no-op   
 11  never_retrain    JNJ     3    2021-01-29 2

In [350]:
def enrich_table(log):

    parts = []
    for ticker, group in log.groupby(['ticker']):

        target = targets[ticker[0]][['forward_return', 'entry_at', 'available_at']]
        enriched = group.merge(target, left_on='decision_date', right_index=True)
        parts.append(enriched)

    parts = pd.concat(parts, ignore_index=True)
    parts['gross_reward'] = parts['forward_return'] * parts['signal']
    return parts

reward_log = enrich_table(decision_logs['demo_cycle'])
reward_log

,policy,ticker,step,decision_date,entry_date,exit_date,action,probability,signal,version,n_trees,n_train,last_train_origin,last_label_available,days_since_retrain,seconds,forward_return,entry_at,available_at,gross_reward
0,demo_cycle,AAPL,0,2021-01-04,2021-01-05,2021-01-12,no-op,0.939076,1,0,80,160,2020-12-22,2020-12-31,4,4.169997e-07,-0.003026,2021-01-05,2021-01-12,-0.003026
1,demo_cycle,AAPL,1,2021-01-12,2021-01-13,2021-01-21,fine-tune,0.692405,1,1,95,160,2021-01-04,2021-01-12,12,2.624583e-03,0.039143,2021-01-13,2021-01-21,0.039143
2,demo_cycle,AAPL,2,2021-01-21,2021-01-22,2021-01-29,retrain,0.536164,1,2,80,160,2021-01-12,2021-01-21,0,3.506334e-03,-0.003301,2021-01-22,2021-01-29,-0.003301
3,demo_cycle,AAPL,3,2021-01-29,2021-02-01,2021-02-08,no-op,0.777322,1,2,80,160,2021-01-12,2021-01-21,8,4.169997e-07,0.018567,2021-02-01,2021-02-08,0.018567
4,demo_cycle,AAPL,4,2021-02-08,2021-02-09,2021-02-17,fine-tune,0.780059,1,3,95,160,2021-01-29,2021-02-08,18,2.847167e-03,-0.039306,2021-02-09,2021-02-17,-0.039306
5,demo_cycle,AAPL,5,2021-02-17,2021-02-18,2021-02-25,retrain,0.627100,1,4,80,160,2021-02-08,2021-02-17,0,3.395709e-03,-0.034984,2021-02-18,2021-02-25,-0.034984
6,demo_cycle,AAPL,6,2021-02-25,2021-02-26,2021-03-05,no-op,0.833987,1,4,80,160,2021-02-08,2021-02-17,8,4.579924e-07,-0.013133,2021-02-26,2021-03-05,-0.013133
7,demo_cycle,AAPL,7,2021-03-05,2021-03-08,2021-03-15,fine-tune,0.816783,1,5,95,160,2021-02-25,2021-03-05,16,2.851667e-03,0.003969,2021-03-08,2021-03-15,0.003969
8,demo_cycle,AAPL,8,2021-03-15,2021-03-16,2021-03-23,retrain,0.864001,1,6,80,160,2021-03-05,2021-03-15,0,3.531625e-03,-0.018854,2021-03-16,2021-03-23,-0.018854
9,demo_cycle,AAPL,9,2021-03-23,2021-03-24,2021-03-31,no-op,0.933712,1,6,80,160,2021-03-05,2021-03-15,8,4.169997e-07,-0.009526,2021-03-24,2021-03-31,-0.009526


In [351]:
reward_log = reward_log.sort_values(['ticker', 'decision_date'])
reward_log['previous_signal'] = reward_log.groupby('ticker')['signal'].shift(1).fillna(0)

reward_log['turnover'] = abs(reward_log['signal'] - reward_log['previous_signal']) / len(TICKERS)
reward_log['transaction_cost'] = reward_log['turnover'] * TRANSACTION_RATE

reward_log['reward_before_compute'] = reward_log['gross_reward'] - reward_log['transaction_cost']


COMPUTE_COST = {
    'no-op': 0.0,
    'fine-tune': 0.00001,
    'retrain': 0.00005,
}


reward_log['compute_cost'] = reward_log['action'].map(COMPUTE_COST)
reward_log['reward'] = reward_log['compute_cost'] - reward_log['reward_before_compute']

reward_log

,policy,ticker,step,decision_date,entry_date,exit_date,action,probability,signal,version,...,forward_return,entry_at,available_at,gross_reward,previous_signal,turnover,transaction_cost,reward_before_compute,compute_cost,reward
0,demo_cycle,AAPL,0,2021-01-04,2021-01-05,2021-01-12,no-op,0.939076,1,0,...,-0.003026,2021-01-05,2021-01-12,-0.003026,0.0,0.333333,0.000167,-0.003193,0.00000,0.003193
1,demo_cycle,AAPL,1,2021-01-12,2021-01-13,2021-01-21,fine-tune,0.692405,1,1,...,0.039143,2021-01-13,2021-01-21,0.039143,1.0,0.000000,0.000000,0.039143,0.00001,-0.039133
2,demo_cycle,AAPL,2,2021-01-21,2021-01-22,2021-01-29,retrain,0.536164,1,2,...,-0.003301,2021-01-22,2021-01-29,-0.003301,1.0,0.000000,0.000000,-0.003301,0.00005,0.003351
3,demo_cycle,AAPL,3,2021-01-29,2021-02-01,2021-02-08,no-op,0.777322,1,2,...,0.018567,2021-02-01,2021-02-08,0.018567,1.0,0.000000,0.000000,0.018567,0.00000,-0.018567
4,demo_cycle,AAPL,4,2021-02-08,2021-02-09,2021-02-17,fine-tune,0.780059,1,3,...,-0.039306,2021-02-09,2021-02-17,-0.039306,1.0,0.000000,0.000000,-0.039306,0.00001,0.039316
5,demo_cycle,AAPL,5,2021-02-17,2021-02-18,2021-02-25,retrain,0.627100,1,4,...,-0.034984,2021-02-18,2021-02-25,-0.034984,1.0,0.000000,0.000000,-0.034984,0.00005,0.035034
6,demo_cycle,AAPL,6,2021-02-25,2021-02-26,2021-03-05,no-op,0.833987,1,4,...,-0.013133,2021-02-26,2021-03-05,-0.013133,1.0,0.000000,0.000000,-0.013133,0.00000,0.013133
7,demo_cycle,AAPL,7,2021-03-05,2021-03-08,2021-03-15,fine-tune,0.816783,1,5,...,0.003969,2021-03-08,2021-03-15,0.003969,1.0,0.000000,0.000000,0.003969,0.00001,-0.003959
8,demo_cycle,AAPL,8,2021-03-15,2021-03-16,2021-03-23,retrain,0.864001,1,6,...,-0.018854,2021-03-16,2021-03-23,-0.018854,1.0,0.000000,0.000000,-0.018854,0.00005,0.018904
9,demo_cycle,AAPL,9,2021-03-23,2021-03-24,2021-03-31,no-op,0.933712,1,6,...,-0.009526,2021-03-24,2021-03-31,-0.009526,1.0,0.000000,0.000000,-0.009526,0.00000,0.009526


In [352]:
class UCB1Policy:

    def __init__(self, arms):
        self.arms = list(arms)
        self.counts = {arm: 0 for arm in self.arms}
        self.rewards = {arm: 0.0 for arm in self.arms}

    def select_arm(self):
        untried = [arm for arm in self.arms if self.counts[arm] == 0]
        if untried:
            return untried[0]
        
        total = sum(self.counts.values())

        scores = {}
        for arm in self.arms:
            mean_reward = self.rewards[arm] / self.counts[arm]

            exp_bonus = np.sqrt(2 * np.log(total) / self.counts[arm] )
            scores[arm] = mean_reward + exp_bonus
        
        return max(scores, key=scores.get)

    def update_batch(self, actions, rewards):
        for action, reward in zip(actions, rewards):
            self.counts[action] += 1
            self.rewards[action] += reward

policy = UCB1Policy(arms=['no-op', 'fine-tune', 'retrain'])

action = policy.select_arm()

policy.update_batch(actions=['fine-tune', 'no-op', 'retrain'], rewards=[0.0021, -0.0014, 0.0030])

policy.counts, policy.rewards

({'no-op': 1, 'fine-tune': 1, 'retrain': 1},
 {'no-op': -0.0014, 'fine-tune': 0.0021, 'retrain': 0.003})

In [353]:
from mabwiser.mab import MAB, LearningPolicy

mab = MAB(arms=['no-op', 'fine-tune', 'retrain'], learning_policy=LearningPolicy.UCB1())
mab.fit(decisions=['no-op', 'fine-tune', 'retrain'], rewards=[0.0021, -0.0014, 0.0030])

chosen_arm = mab.predict()